In [10]:
import requests
import json
import os
import time
from datetime import datetime

# API URLs
TOP_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

headers = {"User-Agent": "TrendPulse/1.0"}

# categories with improved keywords
categories = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "startup", "app"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "match"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "experiment"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "streaming", "tv"]
}


# function to assign category
def get_category(title):
    title = title.lower()

    for cat in categories:
        for word in categories[cat]:
            if word in title:
                return cat

    return "technology"   # fallback so no data is lost


# fetch top story IDs
def get_top_ids():
    try:
        res = requests.get(TOP_URL, headers=headers)
        return res.json()[:1000]   # fetch more stories
    except:
        print("Error fetching IDs")
        return []


# fetch each story
def get_story(sid):
    try:
        res = requests.get(ITEM_URL.format(sid), headers=headers)
        return res.json()
    except:
        print(f"Skipping {sid}")
        return None


def main():
    ids = get_top_ids()

    data = []
    count = {c: 0 for c in categories}

    print("Starting collection...\n")

    for sid in ids:
        story = get_story(sid)

        if not story or "title" not in story:
            continue

        title = story["title"]
        cat = get_category(title)

        # create record
        item = {
            "post_id": story.get("id"),
            "title": title,
            "category": cat,
            "score": story.get("score", 0),
            "num_comments": story.get("descendants", 0),
            "author": story.get("by", "unknown"),
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        data.append(item)
        count[cat] += 1

        print(f"{len(data)} collected -> {cat}")

        # stop when enough stories collected
        if len(data) >= 120:
            break

    # create data folder
    os.makedirs("data", exist_ok=True)

    filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

    with open(filename, "w") as f:
        json.dump(data, f, indent=4)

    print("\nFinal category counts:")
    print(count)

    print(f"\nCollected {len(data)} stories")
    print(f"Saved to {filename}")


if __name__ == "__main__":
    main()

Starting collection...

1 collected -> technology
2 collected -> technology
3 collected -> technology
4 collected -> technology
5 collected -> technology
6 collected -> technology
7 collected -> technology
8 collected -> technology
9 collected -> technology
10 collected -> technology
11 collected -> technology
12 collected -> technology
13 collected -> technology
14 collected -> technology
15 collected -> technology
16 collected -> technology
17 collected -> technology
18 collected -> technology
19 collected -> worldnews
20 collected -> technology
21 collected -> technology
22 collected -> technology
23 collected -> technology
24 collected -> technology
25 collected -> technology
26 collected -> technology
27 collected -> technology
28 collected -> technology
29 collected -> technology
30 collected -> technology
31 collected -> technology
32 collected -> technology
33 collected -> sports
34 collected -> technology
35 collected -> technology
36 collected -> technology
37 collected -> te

In [11]:
import os
os.listdir("data")

['trends_20260414.json']

In [12]:
import json

with open("data/trends_20260414.json") as f:
    data = json.load(f)

print(len(data))        # should be 100+
print(data[0])          # check structure

120
{'post_id': 47762864, 'title': 'Backblaze has stopped backing up your data', 'category': 'technology', 'score': 88, 'num_comments': 35, 'author': 'rrreese', 'collected_at': '2026-04-14 10:11:02'}


In [13]:
from google.colab import files
files.download("data/trends_20260414.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>